In [16]:
# --------------------------------
# Imports & Path Setup
# --------------------------------

%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import matplotlib.pyplot as plt
from PIL import Image
import torch

from src.compression.fourier import FourierCompressor
from src.compression.wavelet import WaveletCompressor
from src.utils.preprocessing import CompressionDataset, get_dataloaders

import os
os.getcwd()  

'/hpc/home/efavale/deep-image-restoration'

In [17]:
# --------------------------------
# Parameters
# --------------------------------

RAW_DIR = Path("data/raw")
RESIZED_DIR = Path("data/resized")
COMP_DIR = Path("data/compressed")

IMAGE_SIZE = 256
BATCH_SIZE = 8
SPLITS = (0.70, 0.15, 0.15)
SEED = 42

In [18]:
# --------------------------------
# Resize & Save
# --------------------------------

RESIZED_DIR.mkdir(exist_ok=True)

raw_files = sorted(RAW_DIR.glob("*"))
raw_files = [f for f in raw_files if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]

print(f"Images found: {len(raw_files)}")

for path in raw_files:
    img = Image.open(path).convert("RGB")
    resized = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
    out = RESIZED_DIR / (path.stem + ".png")
    resized.save(out)

print(f"Resized images saved in: {RESIZED_DIR}")

Images found: 500


Resized images saved in: data/resized


In [19]:
# --------------------------------
# Compress & Save
# --------------------------------

COMP_DIR.mkdir(exist_ok=True)
fourier = FourierCompressor(keep_fraction=0.25)
wavelet = WaveletCompressor(wavelet="haar", level=1, threshold=20.0, keep_ll_only=True)

resized_files = sorted(RESIZED_DIR.glob("*.png"))
print(f"Images to compress: {len(resized_files)}")

for input_path in RESIZED_DIR.glob("*.png"):
    fourier.compress(input_image_path=str(input_path), output_dir=str(COMP_DIR))
    wavelet.compress(input_image_path=str(input_path), output_dir=str(COMP_DIR))
    print(f"Processed: {input_path.name}")

print(f"Compressed images saved in: {COMP_DIR}")

Images to compress: 500
Processed: 100007.png
Processed: 100039.png
Processed: 100075.png
Processed: 100080.png
Processed: 100098.png
Processed: 100099.png
Processed: 10081.png
Processed: 101027.png
Processed: 101084.png
Processed: 101085.png
Processed: 101087.png
Processed: 102061.png
Processed: 102062.png
Processed: 103006.png
Processed: 103029.png
Processed: 103041.png
Processed: 103070.png
Processed: 103078.png
Processed: 104010.png
Processed: 104022.png
Processed: 104055.png
Processed: 105019.png
Processed: 105025.png
Processed: 105027.png
Processed: 105053.png
Processed: 106005.png
Processed: 106020.png
Processed: 106024.png
Processed: 106025.png
Processed: 106047.png
Processed: 107014.png
Processed: 107045.png
Processed: 107072.png
Processed: 108004.png
Processed: 108005.png
Processed: 108036.png
Processed: 108041.png
Processed: 108069.png
Processed: 108070.png
Processed: 108073.png
Processed: 108082.png
Processed: 109034.png
Processed: 109053.png
Processed: 109055.png
Processed

In [20]:
# --------------------------------
# Sanity Check: shape consistency
# --------------------------------

errors = []
for comp_path in sorted(COMP_DIR.glob("*.png")):
    name = comp_path.name

    if "fourier" in name:
        raw_id = comp_path.stem.replace("_fourier_25", "")
    elif "wavelet" in name:
        raw_id = comp_path.stem.replace("_wavelet_haar_1_20_ll_only", "")
    else:
        errors.append(f"Unknown compression type: {name}")
        continue

    raw_path = RESIZED_DIR / (raw_id + ".png")

    if not raw_path.exists():
        errors.append(f"Missing raw: {raw_id}")
        continue

    comp = Image.open(comp_path)
    orig = Image.open(raw_path)

    if comp.size != orig.size:
        errors.append(f"Size mismatch: {raw_id} — {comp.size} vs {orig.size}")

if errors:
    for e in errors:
        print(f"✗ {e}")
else:
    print(f"✓ All {len(list(COMP_DIR.glob('*.png')))} pairs are consistent")

✓ All 1000 pairs are consistent


In [21]:
# --------------------------------
# Dataset & DataLoaders
# --------------------------------

dataset = CompressionDataset(
    input_dir=RESIZED_DIR,
    compressed_dir=COMP_DIR,
    image_size=IMAGE_SIZE,
    suffixs=["_fourier_25", "_wavelet_haar_1_20_ll_only"],
)

train_loader, val_loader, test_loader = get_dataloaders(
    dataset,
    splits=SPLITS,
    batch_size=BATCH_SIZE,
    seed=SEED,
    num_workers=0,
)

print(f"Total : {len(dataset)}")
print(f"Train : {len(train_loader.dataset)}")
print(f"Val : {len(val_loader.dataset)}")
print(f"Test : {len(test_loader.dataset)}")

Total : 1000
Train : 700
Val : 150
Test : 150


In [22]:
# --------------------------------
# Tensor Shape Check
# --------------------------------

comp, orig = dataset[0]

print(f"Shape : {comp.shape}")
print(f"Dtype : {comp.dtype}")
print(f"Range : [{comp.min():.3f}, {comp.max():.3f}]")

assert comp.shape == orig.shape
assert comp.shape == torch.Size([3, IMAGE_SIZE, IMAGE_SIZE])
print("Shape check passed")

Shape : torch.Size([3, 256, 256])
Dtype : torch.float32
Range : [0.075, 0.992]
Shape check passed
